# 🏔 cryoDRGN — conformational **landscape** analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ts387/cryodrgn/blob/claude/cryodrgn-colab-notebook-5mf3p3/cryoDRGN_colab_landscape.ipynb)

`cryodrgn analyze_landscape` answers a different question from `cryodrgn analyze`.

`analyze` clusters the **latent vectors**. But the latent's metric is whatever the encoder
happened to learn — two latent clusters can be structurally near-identical, and one latent
cluster can straddle a real conformational change. `analyze_landscape` instead generates a
few hundred to a few thousand volumes, masks them, and runs **PCA and hierarchical clustering
on the voxels**. Distances there are actual density differences, so the states it finds and the
occupancies it reports mean something structural.

| Step | What happens |
|------|--------------|
| 1. Setup | Install cryoDRGN, mount Drive |
| 2. Attach | Pick the run and epoch; resolve Å/px; build the local mirror workdir |
| 3. UMAP | Make the `analyze.N/umap.pkl` that landscape requires |
| 4. Landscape | Generate volumes → mask → volume PCA → cluster into states |
| 5. View | Mask slices, volume-PCA plots, state occupancies, states on the UMAP |
| 6. States | Per-state particle indices, ready for re-extraction or a fresh model |
| 7. Save | Copy the results you want back to Drive |
| 8. Full | *(optional)* `analyze_landscape_full` — landscape coordinates for **every** particle |

### This one needs a GPU

Unlike `analyze --skip-vol`, landscape's whole method is generating volumes — 1000 of them by
default. Budget accordingly; a T4 is workable, an L4/A100 noticeably quicker.

### Three things that catch people out

**`--Apix` defaults to 1 and is *not* inferred from `ctf.pkl`** — unlike `cryodrgn analyze`,
which resolves it (`analyze.py:462-488`). This is not merely a header cosmetic: `--dilate` and
`--cosine-edge` are given in **Ångströms** and converted with `int(dilation // apix)`
(`masking.py:110`), so a wrong Å/px silently rescales the mask. Cell 2.2 resolves the real value
from `ctf.pkl` and cell 4.1 passes it.

**It requires `<workdir>/analyze.<epoch>/umap.pkl`** and raises if it is missing — the path is
hardcoded to the workdir, ignoring whatever `-o` you gave `analyze`
(`analyze_landscape.py:626-635`). Cell 3.1 puts one where it is wanted.

**The volumes are big.** 1000 volumes at box 128 is ~8.4 GB. This notebook works in local disk
and lets you choose what to copy to Drive in cell 7.1.

## 1 · Setup

In [ ]:
#@title 1.1 · Check the GPU runtime { display-mode: "form" }
#@markdown Confirms a CUDA GPU is attached. If this prints **"No GPU found"**, go to
#@markdown **Runtime → Change runtime type → GPU** and re-run this cell.
import subprocess, sys

print("=" * 60)
gpu = subprocess.run(["nvidia-smi",
                      "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode == 0 and gpu.stdout.strip():
    name, mem, driver = [x.strip() for x in gpu.stdout.strip().split(",")]
    print(f"✅ GPU detected : {name}")
    print(f"   Memory       : {mem}")
    print(f"   Driver       : {driver}")
else:
    print("❌ No GPU found!")
    print("   Runtime → Change runtime type → Hardware accelerator → GPU,")
    print("   then re-run this cell. cryoDRGN training needs a GPU.")
print("=" * 60)

In [ ]:
#@title 1.2 · Install cryoDRGN { display-mode: "form" }
#@markdown Installs cryoDRGN from PyPI. Colab's pre-installed PyTorch/CUDA are kept.
#@markdown <br>• **stable** – the recommended release &nbsp;•&nbsp; **beta** – newest dev build from TestPyPI
release_channel = "stable"  #@param ["stable", "beta"]
#@markdown Optionally pin an exact version (e.g. `4.3.0`); leave blank for the latest.
version = ""  #@param {type:"string"}
#@markdown A few dependencies are pinned to versions other than Colab's defaults, so the
#@markdown runtime **restarts automatically** at the end. That is expected — just carry
#@markdown on with the next cell afterwards.
restart_after_install = True  #@param {type:"boolean"}

import subprocess, sys

pkg = "cryodrgn"
if version.strip():
    pkg = f"cryodrgn=={version.strip()}"

if release_channel == "beta":
    cmd = [sys.executable, "-m", "pip", "install", "-q",
           "-i", "https://test.pypi.org/simple/",
           "--extra-index-url", "https://pypi.org/simple/",
           "cryodrgn", "--pre"]
    if version.strip():
        cmd[cmd.index("cryodrgn")] = pkg
else:
    cmd = [sys.executable, "-m", "pip", "install", "-q", pkg]

print("Installing", pkg, f"({release_channel} channel) — this takes ~1-2 min...\n")
ret = subprocess.run(cmd)
if ret.returncode != 0:
    raise SystemExit("❌ pip install failed — see the log above.")

# --- realign torchvision with torch -------------------------------------------------
# cryoDRGN pins torch<2.10, so pip may DOWNGRADE Colab's torch. Colab's pre-installed
# torchvision was compiled against the newer torch, and once they disagree importing it
# raises "operator torchvision::nms does not exist". That breaks EVERY cryodrgn command,
# because the CLI eagerly imports all command modules and analyze_landscape_full imports
# umap -> torchvision. Matching pair is torch 2.N <-> torchvision 0.(N+15).
import importlib.metadata as md

def _ver(p):
    try:
        return md.version(p)
    except md.PackageNotFoundError:
        return None

tver, tvver = _ver("torch"), _ver("torchvision")
if tver and tvver:
    tmaj, tmin = (int(x) for x in tver.split(".")[:2])
    tvmin = int(tvver.split(".")[1])
    want = tmin + 15 if tmaj == 2 else None
    if want is not None and tvmin != want:
        print(f"\n⚠️  torch {tver} and torchvision {tvver} are incompatible "
              f"(cryoDRGN's torch<2.10 pin downgraded torch).")
        print(f"   Installing torchvision 0.{want}.* to match...")
        fix = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "--no-deps",
             f"torchvision==0.{want}.*"])
        if fix.returncode == 0:
            print(f"   ✅ torchvision realigned to 0.{want}.*")
        else:
            print(f"   ❌ Could not install torchvision 0.{want}.* — if cryodrgn commands "
                  f"fail with 'torchvision::nms does not exist', run:")
            print(f"      !pip install --no-deps 'torchvision==0.{want}.*'")

print("\n✅ cryoDRGN installed.")
if restart_after_install:
    print("🔄 Restarting the runtime to finalize the install (this is normal)...")
    print("   When it reconnects, continue from cell 1.3 — do NOT re-run this cell.")
    get_ipython().kernel.do_shutdown(True)

In [ ]:
#@title 1.3 · Verify the installation { display-mode: "form" }
#@markdown Run this **after** the runtime has restarted. The `cryodrgn --version` smoke-test is
#@markdown the important one: the CLI imports *every* command module on startup, so a broken
#@markdown dependency anywhere makes all commands fail — better to catch it here than mid-run.
import sys, subprocess
import torch, cryodrgn

print(f"cryoDRGN version : {cryodrgn.__version__}")
print(f"PyTorch version  : {torch.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device      : {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  CUDA not available — fine for Step 4 (CPU-only), but Step 6 needs a GPU (cell 1.1).")

# torchvision must match torch or `import umap` blows up inside the cryodrgn CLI
try:
    import torchvision
    print(f"torchvision      : {torchvision.__version__} (ok)")
except Exception as e:
    msg = str(e).splitlines()[0]
    want = "0.%d.*" % (int(torch.__version__.split(".")[1]) + 15)
    if "numpy.dtype size changed" in msg or "binary incompatibility" in msg:
        # cryoDRGN pins numpy<1.27, downgrading Colab's numpy 2.x. C extensions that were
        # compiled against numpy 2.x headers then fail their ABI check on import.
        print(f"⚠️  torchvision fails a NumPy ABI check: {msg}")
        print("   Cause: cryoDRGN pins numpy<1.27, so Colab's numpy 2.x was downgraded and")
        print("   torchvision (built against numpy 2.x) no longer matches.")
        print("   This is USUALLY HARMLESS: cryoDRGN never imports torchvision itself — only")
        print("   umap does, and cryodrgn.analysis imports umap lazily. Every cryodrgn command")
        print("   also runs as a subprocess. Treat the CLI check below as the real verdict, and")
        print("   don't 'fix' this unless something actually fails.")
    else:
        print(f"❌ torchvision is broken: {msg}")
        print("   Looks like a torch/torchvision version mismatch rather than a NumPy issue.")
        print(f"   Fix with:  !pip install --no-deps 'torchvision=={want}'")
        print("   then re-run this cell (no restart needed).")

print("\n$ cryodrgn --version")
r = subprocess.run(["cryodrgn", "--version"], capture_output=True, text=True)
print((r.stdout + r.stderr).strip()[-2000:])
if r.returncode != 0:
    raise RuntimeError("The cryodrgn CLI failed to start — fix the error above before continuing.")
print("\n✅ CLI healthy — all command modules import cleanly.")

In [ ]:
#@title 1.4 · Mount Google Drive { display-mode: "form" }
#@markdown Click the link that appears, pick your Google account, and paste the code
#@markdown (or approve the pop-up). Your Drive appears under `/content/drive/MyDrive`.
from google.colab import drive
drive.mount("/content/drive")
print("\n✅ Drive mounted at /content/drive/MyDrive")

## 2 · Attach to a trained model

Everything runs against a **local mirror** of the run folder rather than the Drive folder
itself. That keeps the ~GB of volume traffic off Drive, leaves the training run untouched, and
lets cell 3.1 place `analyze.N/umap.pkl` exactly where `analyze_landscape` insists on finding it
without writing into your model directory.

In [ ]:
#@title 2.1 · Point at the run { display-mode: "form" }
#@markdown Drive project folder — the one holding your model directories.
drive_project_dir = "/content/drive/MyDrive/cryodrgn_project"  #@param {type:"string"}
#@markdown Model folder name inside it, e.g. `01_cryodrgn128_filtered`.
run_name = ""  #@param {type:"string"}
#@markdown Epoch to analyze. **-1** uses the highest `z.N.pkl` present.
epoch = -1  #@param {type:"integer"}

import os, re, glob
if not run_name.strip():
    raise ValueError("Set run_name to the model folder you want to analyze.")
DRIVE_DIR = os.path.abspath(drive_project_dir)
RUN = os.path.join(DRIVE_DIR, run_name.strip())
if not os.path.isdir(RUN):
    raise FileNotFoundError(f"{RUN} not found — check drive_project_dir / run_name.")
if not os.path.exists(os.path.join(RUN, "config.yaml")):
    raise FileNotFoundError(f"No config.yaml in {RUN} — is this a cryoDRGN run folder?")

eps = sorted(int(m.group(1)) for p in glob.glob(os.path.join(RUN, "z.*.pkl"))
             for m in [re.search(r"z\.(\d+)\.pkl$", os.path.basename(p))]
             if m and int(m.group(1)) > 0)
if not eps:
    raise FileNotFoundError(f"No numbered z.N.pkl in {RUN}.")
EP = int(epoch) if int(epoch) > 0 else max(eps)
if EP not in eps:
    raise FileNotFoundError(f"No z.{EP}.pkl in {RUN}. Available epochs: {eps}")
if not os.path.exists(os.path.join(RUN, f"weights.{EP}.pkl")):
    raise FileNotFoundError(f"z.{EP}.pkl exists but weights.{EP}.pkl does not — landscape "
                            f"needs the decoder weights to generate volumes.")

for k, v in dict(LS_DRIVE=DRIVE_DIR, LS_RUN=RUN, LS_EPOCH=str(EP),
                 LS_NAME=run_name.strip()).items():
    os.environ[k] = v
print(f"run       : {RUN}")
print(f"epochs    : {eps}")
print(f"analyzing : epoch {EP}")

In [ ]:
#@title 2.2 · Resolve Å/px and build the mirror workdir { display-mode: "form" }
#@markdown Pixel size. **`0` resolves it from `ctf.pkl`**, exactly as `cryodrgn analyze` does
#@markdown (`analyze.py:475-488`) — `analyze_landscape` itself would silently use 1.0.
apix = 0  #@param {type:"number"}
#@markdown Where to do the work. Local disk is much faster than Drive for thousands of volumes.
work_root = "/content/landscape_work"  #@param {type:"string"}

import os, shutil
import numpy as np
import yaml
from cryodrgn import utils

RUN, EP = os.environ["LS_RUN"], int(os.environ["LS_EPOCH"])
cfg = yaml.safe_load(open(os.path.join(RUN, "config.yaml")))
D_box = cfg["lattice_args"]["D"] - 1
zdim = cfg["model_args"]["zdim"]

# --- Å/px -----------------------------------------------------------------------------
# analyze_landscape's --Apix defaults to 1 and it never looks at ctf.pkl. That value ends up
# in every volume header AND in the mask arithmetic, since --dilate/--cosine-edge are in
# Angstroms (masking.py:110). So resolve it here the way `cryodrgn analyze` does.
APIX = float(apix)
if APIX <= 0:
    _ctf = cfg["dataset_args"].get("ctf")
    if _ctf and os.path.exists(_ctf):
        cp = np.asarray(utils.load_pkl(_ctf))
        aps, szs = set(cp[:, 1]), set(cp[:, 0])
        if len(aps) == 1:
            # cast out of float32 before the arithmetic, or the rounding cannot land on a
            # clean value and the number printed here differs from the one passed to --Apix
            _ap, _sz = float(tuple(aps)[0]), float(tuple(szs)[0])
            APIX = round(_ap * _sz / D_box, 6)
            print(f"A/px      : {APIX:g} (from {os.path.basename(_ctf)}: "
                  f"{_ap:g} A/px at box {_sz:g} -> box {D_box})")
        else:
            raise ValueError(
                f"{_ctf} has {len(aps)} distinct pixel sizes (multiple optics groups). "
                f"cryoDRGN cannot resolve one — set apix above explicitly.")
    else:
        raise FileNotFoundError(
            f"config.yaml points at ctf={_ctf!r}, which is not readable, so A/px cannot be\n"
            f"  resolved. Set apix above. Do NOT leave it to analyze_landscape's default of\n"
            f"  1.0 — that silently rescales --dilate and --cosine-edge.")
else:
    print(f"A/px      : {APIX} (set explicitly)")

# --- mirror workdir -------------------------------------------------------------------
WORK = os.path.abspath(os.path.join(work_root, f"{os.environ['LS_NAME']}.{EP}"))
os.makedirs(WORK, exist_ok=True)
shutil.copyfile(os.path.join(RUN, "config.yaml"), os.path.join(WORK, "config.yaml"))
# cryodrgn analyze (cell 3.1) opens <workdir>/run.log unguarded via analysis.parse_loss
# (analysis.py:31). A finished run has one; a run whose session died may not, so make sure
# something is there rather than crashing after the slow UMAP step.
_srclog, _dstlog = os.path.join(RUN, "run.log"), os.path.join(WORK, "run.log")
if os.path.exists(_srclog):
    shutil.copyfile(_srclog, _dstlog)
elif not os.path.exists(_dstlog):
    open(_dstlog, "w").close()
    print("run.log   : not present in the run folder — using an empty one so analyze cannot")
    print("            crash on it. Its learning-curve PNG will be blank; nothing else is hit.")
for f in (f"z.{EP}.pkl", f"weights.{EP}.pkl"):
    s, d = os.path.join(RUN, f), os.path.join(WORK, f)
    if not os.path.exists(d) or os.path.getsize(d) != os.path.getsize(s):
        print(f"staging   : {f} ({os.path.getsize(s)/2**20:.0f} MB)", flush=True)
        shutil.copyfile(s, d)

z = np.asarray(utils.load_pkl(os.path.join(WORK, f"z.{EP}.pkl")))
free = shutil.disk_usage(WORK).free
os.environ.update(LS_WORK=WORK, LS_APIX=str(APIX), LS_D=str(D_box))
print(f"particles : {z.shape[0]:,}   zdim {zdim}   box {D_box}")
print(f"workdir   : {WORK}   ({free/2**30:.0f} GiB free)")
_ind = cfg["dataset_args"].get("ind")
if _ind:
    print(f"--ind     : {_ind}")
    print("            state particle indices will be mapped back to the ORIGINAL numbering")
    print("            (analyze_landscape.py:648-653), so they index your full stack.")
#@markdown ---
#@markdown Volume storage: `sketch_size × downsample³ × 4` bytes. Cell 4.1 prints the figure
#@markdown for your settings and refuses to start if the disk cannot hold it.

## 3 · The UMAP prerequisite

`analyze_landscape` reads `<workdir>/analyze.<epoch>/umap.pkl` and raises if it is absent — it
uses that embedding as the backdrop for plotting where each state sits in the latent
(`analyze_landscape.py:626-635`). The path is hardcoded to the workdir, so a UMAP you made
elsewhere (say with the live-analysis notebook, which writes to its own folder) will not be
found unless it is copied in.

This cell either reuses an existing one or makes it with `cryodrgn analyze --skip-vol`, which is
CPU-only and needs no GPU. UMAP on several hundred thousand particles takes a few minutes.

In [ ]:
#@title 3.1 · Get analyze.N/umap.pkl into place { display-mode: "form" }
#@markdown Reuse a `umap.pkl` from somewhere else instead of computing one — e.g.
#@markdown `.../live_analysis/<run>/analyze.11/umap.pkl`. Blank = look in the run folder,
#@markdown then compute.
umap_pkl = ""  #@param {type:"string"}
#@markdown Recompute even if one is already in place.
force_recompute = False  #@param {type:"boolean"}

import os, shutil, time
import numpy as np
from cryodrgn import utils

RUN, WORK, EP = os.environ["LS_RUN"], os.environ["LS_WORK"], int(os.environ["LS_EPOCH"])
adir = os.path.join(WORK, f"analyze.{EP}")
os.makedirs(adir, exist_ok=True)
dst = os.path.join(adir, "umap.pkl")

def _ok(p):
    """A umap.pkl landscape can actually use: (N, 2) and matching the latent."""
    try:
        u = np.asarray(utils.load_pkl(p))
    except Exception:
        return None
    n = np.asarray(utils.load_pkl(os.path.join(WORK, f"z.{EP}.pkl"))).shape[0]
    if u.ndim != 2 or u.shape[1] != 2:
        print(f"   {p}: shape {u.shape}, expected (N, 2) — ignoring")
        return None
    if u.shape[0] != n:
        print(f"   {p}: {u.shape[0]:,} rows but z.{EP}.pkl has {n:,} — a different epoch or "
              f"run; ignoring")
        return None
    return u.shape[0]

src = None
if umap_pkl.strip():
    src = os.path.abspath(umap_pkl.strip())
    if not os.path.exists(src):
        raise FileNotFoundError(f"{src} not found.")
elif os.path.exists(os.path.join(RUN, f"analyze.{EP}", "umap.pkl")):
    src = os.path.join(RUN, f"analyze.{EP}", "umap.pkl")
    print(f"found     : {src}")

if force_recompute:
    src = None
if src and _ok(src):
    shutil.copyfile(src, dst)
    print(f"✅ using {src}\n   -> {dst}")
elif not force_recompute and os.path.exists(dst) and _ok(dst):
    print(f"✅ already in place: {dst}")
else:
    print("Computing the UMAP with `cryodrgn analyze --skip-vol` (CPU-only, a few minutes).")
    cmd = f'cryodrgn analyze "{WORK}" {EP} -o "{adir}" --skip-vol --ksample 20'
    print("$", cmd, "\n" + "=" * 70)
    t0 = time.time()
    get_ipython().system(cmd)
    if get_ipython().user_ns.get("_exit_code", 0):
        raise RuntimeError("cryodrgn analyze failed — see the log above.")
    print("=" * 70 + f"\n✅ {(time.time()-t0)/60:.1f} min")

n = _ok(dst)
if not n:
    raise RuntimeError(f"{dst} is still missing or unusable — landscape cannot run.")
print(f"umap.pkl  : {n:,} rows ✓  (analyze_landscape will find it at {dst})")

## 4 · Run the landscape analysis

What the parameters actually control:

| Flag | Meaning | Note |
|---|---|---|
| `-N` sketch_size | volumes generated from k-means centres of `z` | the cost driver, and the resolution of the landscape |
| `-d` downsample | box size of those volumes | 128 is plenty for clustering; smaller is much cheaper |
| `--thresh` | density cutoff for the mask | blank = mean of each volume's 99.99th percentile ÷ 2 |
| `--dilate` | grow the mask, **in Å** | converted with `int(dilate // Apix)` |
| `--cosine-edge` | soft edge width, **in Å** | 0 = hard edge |
| `--pc-dim` | components kept in the volume PCA | |
| `-M` | number of states from agglomerative clustering | the answer changes with this — try a few |
| `--linkage` | `average` or `ward` | `ward` favours equal-sized states |

Cell 3 is a hard prerequisite. Volume generation dominates the runtime; masking, PCA and
clustering together take a couple of minutes.

In [ ]:
#@title 4.1 · analyze_landscape { display-mode: "form" }
#@markdown Volumes to generate. 1000 is the cryoDRGN default; 500 is a reasonable first pass.
sketch_size = 1000  #@param {type:"integer"}
#@markdown Box size for the generated volumes.
downsample = 128  #@param [64, 128, 192, 200, 256] {type:"raw"}
#@markdown Number of states (agglomerative clusters over the volumes).
n_states = 10  #@param {type:"integer"}
#@markdown Linkage for that clustering.
linkage = "average"  #@param ["average", "ward", "complete", "single"]
#@markdown Components kept in the volume PCA, and how many to plot.
pc_dim = 20  #@param {type:"integer"}
plot_dim = 5  #@param {type:"integer"}
#@markdown Mask dilation and cosine edge, **in Ångströms**.
dilate_A = 5  #@param {type:"number"}
cosine_edge_A = 0  #@param {type:"number"}
#@markdown Density threshold for the mask — `0` uses cryoDRGN's automatic value.
thresh = 0  #@param {type:"number"}
#@markdown Optional custom mask (.mrc), which must match the `downsample` box size.
custom_mask = ""  #@param {type:"string"}
#@markdown Reuse volumes from a previous run of this cell instead of regenerating them.
skip_volume_generation = False  #@param {type:"boolean"}
#@markdown Overwrite an existing landscape directory for this epoch.
overwrite = False  #@param {type:"boolean"}

import os, glob, shutil, time
WORK, EP = os.environ["LS_WORK"], int(os.environ["LS_EPOCH"])
APIX, D_box = float(os.environ["LS_APIX"]), int(os.environ["LS_D"])
LAND = os.path.join(WORK, f"landscape.{EP}")

K, dsamp, M = int(sketch_size), int(downsample), int(n_states)
if M < 2:
    raise ValueError("n_states must be at least 2.")
if M > K:
    raise ValueError(f"n_states ({M}) cannot exceed sketch_size ({K}).")
if dsamp > D_box:
    raise ValueError(f"downsample ({dsamp}) exceeds the model's box size ({D_box}); "
                     f"analyze_landscape can only downsample.")
if int(pc_dim) > K:
    raise ValueError(f"pc_dim ({pc_dim}) cannot exceed sketch_size ({K}) — PCA cannot keep "
                     f"more components than it has samples.")
if int(plot_dim) > int(pc_dim):
    raise ValueError(f"plot_dim ({plot_dim}) cannot exceed pc_dim ({pc_dim}) — the plots index "
                     f"pca.components_, which only has pc_dim rows.")

# volumes dominate the disk; refuse rather than fill the disk and die mid-run
need = K * dsamp ** 3 * 4
free = shutil.disk_usage(WORK).free
print(f"volumes   : {K} x {dsamp}^3 float32 = {need/2**30:.1f} GiB   "
      f"({free/2**30:.0f} GiB free)")
if not skip_volume_generation and need > free * 0.9:
    raise RuntimeError(
        f"Not enough local disk: {need/2**30:.1f} GiB needed, {free/2**30:.1f} GiB free.\n"
        f"  Lower sketch_size, or lower downsample (cost scales with the CUBE of the box: "
        f"{dsamp}->{dsamp//2} is 8x cheaper).")

if os.path.exists(LAND) and not (overwrite or skip_volume_generation):
    raise FileExistsError(
        f"{LAND} already exists. Tick `overwrite` to redo it, or `skip_volume_generation` to "
        f"re-cluster the volumes already there (much faster when only n_states/linkage change).")
if overwrite and os.path.exists(LAND) and not skip_volume_generation:
    shutil.rmtree(LAND)

# The effective pixel size changes when the volume box is downsampled from the model's box.
apix_out = round(APIX * D_box / dsamp, 6)
if dsamp != D_box:
    print(f"A/px      : {APIX} at box {D_box} -> {apix_out} at box {dsamp}")
else:
    print(f"A/px      : {apix_out}")
print(f"mask      : dilate {dilate_A} A = {int(dilate_A // apix_out)} px, "
      f"cosine edge {cosine_edge_A} A = {cosine_edge_A / apix_out:.1f} px")
if dilate_A > 0 and int(dilate_A // apix_out) == 0:
    print("            ⚠️  dilation rounds to 0 px at this pixel size — raise dilate_A "
          f"above {apix_out:g} A for any effect.")

cmd = (f'cryodrgn analyze_landscape "{WORK}" {EP} -o "{LAND}" '
       f'-N {K} -d {dsamp} --Apix {apix_out} -M {M} --linkage {linkage} '
       f'--pc-dim {int(pc_dim)} --plot-dim {int(plot_dim)} '
       f'--dilate {int(dilate_A)} --cosine-edge {int(cosine_edge_A)}')
if float(thresh) > 0:
    cmd += f" --thresh {float(thresh)}"
if custom_mask.strip():
    _m = os.path.abspath(custom_mask.strip())
    if not os.path.exists(_m):
        raise FileNotFoundError(f"custom mask not found: {_m}")
    cmd += f' --mask "{_m}"'
if skip_volume_generation:
    if not os.path.isdir(os.path.join(LAND, f"kmeans{K}")):
        raise FileNotFoundError(
            f"skip_volume_generation is on but {LAND}/kmeans{K} does not exist. Either turn it "
            f"off, or set sketch_size back to whatever produced the volumes that are there: "
            f"{sorted(os.path.basename(p) for p in glob.glob(os.path.join(LAND, 'kmeans*')))}")
    cmd += " --skip-vol --skip-umap"

print(f"output    : {LAND}")
print("$", cmd, "\n" + "=" * 70)
t0 = time.time()
get_ipython().system(cmd)
if get_ipython().user_ns.get("_exit_code", 0):
    raise RuntimeError("analyze_landscape failed — see the log above.")
print("=" * 70 + f"\n✅ done in {(time.time()-t0)/60:.1f} min → {LAND}")
os.environ.update(LS_LAND=LAND, LS_K=str(K), LS_M=str(M), LS_LINKAGE=linkage,
                  LS_APIX_OUT=str(apix_out))

## 5 · View the results

The plots to read first:

* **`state_particle_counts.png`** — occupancy. How many particles fall in each state. This is
  the number people usually want out of a landscape analysis.
* **`vol_pca_*_annotated_1_2.png`** — the volumes in volume-PCA space, coloured by state and
  labelled with volume number, so you can go and open a specific `vol_NNN.mrc`.
* **`umap.png`** (inside the clustering folder) — where each state sits on the latent UMAP.
  Compare it against `analyze`'s k-means colouring: **states that are separated here but mixed
  on the latent UMAP are exactly what `analyze` alone would have missed.**
* **`mask_slices.png`** — sanity-check the mask before trusting anything else. Too tight and
  you cluster on a fragment; too loose and you cluster on solvent noise.

In [ ]:
#@title 5.1 · Plots { display-mode: "form" }
#@markdown Width of the displayed images in pixels.
width = 620  #@param {type:"integer"}

import os, glob
from IPython.display import Image, display, Markdown

LAND = os.environ["LS_LAND"]
K, M, LINK = int(os.environ["LS_K"]), int(os.environ["LS_M"]), os.environ["LS_LINKAGE"]
SUB = os.path.join(LAND, f"sketch_clustering_{LINK}_{M}")

def show(path, cap, w=None):
    hits = sorted(glob.glob(path))
    for h in hits[:3]:
        display(Markdown(cap)); display(Image(h, width=w or width))
    return len(hits)

n = 0
display(Markdown(f"## Landscape — epoch {os.environ['LS_EPOCH']}, {K} volumes, {M} states"))
n += show(os.path.join(LAND, "mask_slices.png"),
          "**Mask**, three orthogonal slices. Check it covers the particle and not the solvent.")
n += show(os.path.join(SUB, "state_particle_counts.png"),
          "**Occupancy** — particles per state. The headline result.")
n += show(os.path.join(SUB, "state_volume_counts.png"),
          "Sampled volumes per state (how the sketch was distributed, not occupancy).")
n += show(os.path.join(SUB, f"vol_pca_{K}_1_2.png"),
          "**Volume PCA**, PC1 vs PC2, coloured by state.")
n += show(os.path.join(SUB, "umap.png"),
          "**States on the latent UMAP.** Grey = all particles; coloured = the sketched "
          "volumes. States that separate here but blur together in `analyze`'s k-means view "
          "are what the landscape adds.")
n += show(os.path.join(SUB, f"vol_pca_{K}_annotated_{1}_{2}.png"),
          "Same volume PCA, annotated with volume numbers — use these to open a specific "
          "`kmeans*/vol_NNN.mrc`.", w=900)
n += show(os.path.join(LAND, f"vol_pca_{K}_1_2.png"),
          "Volume PCA before clustering (no state colours).")
print(f"\n{n} plot(s) from {LAND}")
if not n:
    print("Nothing found — has cell 4.1 finished? Check that n_states/linkage here match it.")
avail = sorted(os.path.basename(p) for p in glob.glob(os.path.join(LAND, "sketch_clustering_*")))
print("clusterings present:", avail)

## 6 · States → particle indices

Each state gets `state_XXX_particle_ind.pkl`: the particles whose k-means volume landed in that
state. On a run trained with `--ind`, cryoDRGN maps these back through the original index set
(`analyze_landscape.py:648-653`), so they refer to your **full** stack, not to positions within
the filtered subset — which is what you want for re-extraction or a `.star`/`.cs` export.

Feed one into the export-subset notebook to cut a stack, or into cell 8.3 of the main notebook
to train a model on a single state.

In [ ]:
#@title 6.1 · List states and export their indices { display-mode: "form" }
#@markdown Copy the per-state index files to Drive. Blank = `<project>/landscape_states/<run>.<epoch>`.
export_dir = ""  #@param {type:"string"}
#@markdown Also write a plain `.txt` (one index per line) beside each `.pkl`.
also_txt = True  #@param {type:"boolean"}

import os, glob, re, shutil
import numpy as np
from cryodrgn import utils

LAND, M, LINK = os.environ["LS_LAND"], int(os.environ["LS_M"]), os.environ["LS_LINKAGE"]
DRIVE_DIR, NAME, EP = os.environ["LS_DRIVE"], os.environ["LS_NAME"], os.environ["LS_EPOCH"]
SUB = os.path.join(LAND, f"sketch_clustering_{LINK}_{M}")
OUT = os.path.abspath(export_dir.strip()) if export_dir.strip() \
      else os.path.join(DRIVE_DIR, "landscape_states", f"{NAME}.{EP}")
os.makedirs(OUT, exist_ok=True)

inds = sorted(glob.glob(os.path.join(SUB, "state_*_particle_ind.pkl")))
if not inds:
    raise FileNotFoundError(f"No state_*_particle_ind.pkl in {SUB} — run cell 4.1 first.")

total, rows = 0, []
for p in inds:
    st = re.search(r"state_(\d+)_particle_ind\.pkl$", os.path.basename(p)).group(1)
    sel = np.asarray(utils.load_pkl(p)).ravel()
    total += sel.size
    mean_mrc = os.path.join(SUB, f"state_{st}_mean.mrc")
    rows.append((int(st), sel.size, sel.min() if sel.size else -1,
                 sel.max() if sel.size else -1, os.path.exists(mean_mrc)))
    shutil.copyfile(p, os.path.join(OUT, os.path.basename(p)))
    if also_txt:
        np.savetxt(os.path.join(OUT, os.path.basename(p).replace(".pkl", ".txt")),
                   np.sort(sel), fmt="%d")

print(f"{'state':>6} {'particles':>12} {'share':>8}   index range        mean volume")
for st, n, lo, hi, has in sorted(rows):
    print(f"{st:>6} {n:>12,} {n/total*100:>7.2f}%   {lo:>9,}..{hi:<9,}  "
          f"{'state_%03d_mean.mrc' % st if has else '(missing)'}")
print(f"{'total':>6} {total:>12,}")

allsel = np.concatenate([np.asarray(utils.load_pkl(p)).ravel() for p in inds])
if np.unique(allsel).size != allsel.size:
    print("\n⚠️  the states share particles — they should be disjoint; check n_states/linkage.")
print(f"\nindices → {OUT}")
print("   .pkl feeds `--ind` (main notebook 8.3) and the export-subset notebook directly;")
print("   these are ORIGINAL stack indices even when the run was trained with --ind.")

## 7 · Save to Drive

The landscape directory is mostly volumes — `sketch_size × downsample³ × 4` bytes of them — and
copying all of that to Drive is slow and rarely useful. By default this saves the small,
interpretable outputs: every plot, the PCA and label pickles, the mask, and the per-state mean
and std volumes. The full volume set is opt-in.

The per-state folders contain **symlinks** into `kmeans*/`, not real files. Those are skipped —
Drive's FUSE mount does not support symlinks, and the targets are copied anyway when you ask for
the volumes.

In [ ]:
#@title 7.1 · Copy results to Drive { display-mode: "form" }
#@markdown Blank = `<project>/landscape/<run>.<epoch>`.
dest_dir = ""  #@param {type:"string"}
#@markdown Also copy every generated volume (`kmeans*/vol_*.mrc` and the PC trajectories).
#@markdown This is the multi-GB part.
include_all_volumes = False  #@param {type:"boolean"}

import os, glob, shutil, time
LAND = os.environ["LS_LAND"]
DRIVE_DIR, NAME, EP = os.environ["LS_DRIVE"], os.environ["LS_NAME"], os.environ["LS_EPOCH"]
DEST = os.path.abspath(dest_dir.strip()) if dest_dir.strip() \
       else os.path.join(DRIVE_DIR, "landscape", f"{NAME}.{EP}")

def wanted(rel):
    if rel.endswith((".png", ".pkl", ".txt")):
        return True
    if rel.endswith(".mrc"):
        base = os.path.basename(rel)
        if base in ("mask.mrc",) or base.startswith("state_") or base == "vol_mean.mrc":
            return True
        return bool(include_all_volumes)
    return False

todo, skipped_links, nbytes = [], 0, 0
for root, dirs, files in os.walk(LAND):
    for f in files:
        src = os.path.join(root, f)
        if os.path.islink(src):          # per-state dirs are symlinks into kmeans*/
            skipped_links += 1
            continue
        rel = os.path.relpath(src, LAND)
        if wanted(rel):
            todo.append((src, os.path.join(DEST, rel)))
            nbytes += os.path.getsize(src)

print(f"copying {len(todo)} file(s), {nbytes/2**30:.2f} GiB → {DEST}")
if skipped_links:
    print(f"  ({skipped_links} symlink(s) skipped — they duplicate kmeans volumes)")
if not include_all_volumes:
    vols = len(glob.glob(os.path.join(LAND, "kmeans*", "vol_*.mrc")))
    print(f"  ({vols} generated volume(s) left on local disk — tick include_all_volumes "
          f"to copy them)")
t0 = 0
for i, (s, d) in enumerate(todo):
    os.makedirs(os.path.dirname(d), exist_ok=True)
    if not os.path.exists(d) or os.path.getsize(d) != os.path.getsize(s):
        shutil.copyfile(s, d)
    if i % 50 == 0:
        print(f"  {i}/{len(todo)}", flush=True)
print(f"✅ {DEST}")
print("\n⚠️  Local disk is wiped when this runtime recycles. Anything not copied — including")
print("    the volumes, if you left them — is gone then.")

## 8 · *(optional)* `analyze_landscape_full`

Section 4 gives landscape coordinates to the `-N` sketched volumes only. `analyze_landscape_full`
trains a small MLP to map `z → volume-PCA coordinates`, then applies it to **every particle**, so
each one gets a position in volume space rather than just the sketch. It then clusters with
Leiden (`--resolution`) instead of agglomerative linkage.

It reuses section 4's `mask.mrc` and `vol_pca_obj.pkl` via `--landscape-dir`. Two consequences:

* **`-d` is not yours to choose here.** The code asserts the mask is exactly `(d, d, d)`
  (`analyze_landscape_full.py:228-231`), so the box must match what section 4 used. The cell
  reads it off `mask.mrc` rather than offering a field you could get wrong.
* **Cost is time, not disk.** Its volumes are generated in memory and projected into the PCA
  basis immediately — none are written out (`generate_and_map_volumes`,
  `analyze_landscape_full.py:307-330`). So `-N` costs GPU minutes, not gigabytes. cryoDRGN's
  own default is 10000, which is 10× section 4's work; start lower.

In [ ]:
#@title 8.1 · analyze_landscape_full { display-mode: "form" }
#@markdown Volumes used to train the z → volume-PCA mapping. cryoDRGN's default is `10000`;
#@markdown these are generated in memory, so this costs time rather than disk.
training_volumes = 5000  #@param {type:"integer"}
#@markdown Training epochs, batch size and learning rate for the small MLP
#@markdown (cryoDRGN defaults: 200 / 64 / 1e-4).
epochs = 200  #@param {type:"integer"}
batch_size = 64  #@param {type:"integer"}
lr = 0.0001  #@param {type:"number"}
#@markdown MLP width and depth (cryoDRGN defaults: 512 / 3).
dim = 512  #@param {type:"integer"}
layers = 3  #@param {type:"integer"}
#@markdown Leiden clustering — higher `resolution` gives more, smaller clusters
#@markdown (cryoDRGN defaults: 1.5 / 50).
resolution = 1.5  #@param {type:"number"}
num_neighbors = 50  #@param {type:"integer"}

import os, time
import numpy as np
from cryodrgn import utils
from cryodrgn.source import ImageSource

WORK, EP = os.environ["LS_WORK"], int(os.environ["LS_EPOCH"])
LAND = os.environ.get("LS_LAND")
if not LAND or not os.path.isdir(LAND):
    raise RuntimeError("Run cell 4.1 first — analyze_landscape_full builds on its output.")
for req in ("mask.mrc", "vol_pca_obj.pkl"):
    if not os.path.exists(os.path.join(LAND, req)):
        raise FileNotFoundError(
            f"{LAND}/{req} is missing — analyze_landscape_full needs section 4's mask and PCA "
            f"basis. Re-run cell 4.1.")

# -d must equal the box the section-4 mask was built at, or the run asserts
# (analyze_landscape_full.py:228-231). Read it instead of asking.
mask = np.asarray(ImageSource.from_file(os.path.join(LAND, "mask.mrc")).images().cpu())
if mask.ndim != 3 or len(set(mask.shape)) != 1:
    raise ValueError(f"{LAND}/mask.mrc has shape {mask.shape}; expected a cube.")
dsamp = int(mask.shape[0])
n_particles = np.asarray(utils.load_pkl(os.path.join(WORK, f"z.{EP}.pkl"))).shape[0]
N = int(training_volumes)
if N > n_particles:
    raise ValueError(f"training_volumes ({N:,}) exceeds the {n_particles:,} particles in the "
                     f"run — it samples without replacement.")

print(f"box       : {dsamp} (from mask.mrc — must match section 4, not adjustable here)")
print(f"mask      : {int(mask.astype(bool).sum()):,} voxels")
print(f"volumes   : {N:,} generated in memory, mapped to {dsamp}^3 PCA coordinates")
print(f"            then the trained MLP is applied to all {n_particles:,} particles")

OUT = os.path.join(LAND, "landscape_full")
cmd = (f'cryodrgn analyze_landscape_full "{WORK}" {EP} '
       f'--landscape-dir "{LAND}" -o "{OUT}" '
       f'-N {N} -d {dsamp} '
       f'--epochs {int(epochs)} --batch-size {int(batch_size)} --lr {float(lr)} '
       f'--dim {int(dim)} --layers {int(layers)} '
       f'--resolution {float(resolution)} --num-neighbors {int(num_neighbors)}')
print("$", cmd, "\n" + "=" * 70)
t0 = time.time()
get_ipython().system(cmd)
if get_ipython().user_ns.get("_exit_code", 0):
    raise RuntimeError("analyze_landscape_full failed — see the log above.")
print("=" * 70 + f"\n✅ done in {(time.time()-t0)/60:.1f} min → {OUT}")
print("Outputs: vol_pca_all.pkl (every particle's volume-PCA coordinates),")
print("         umap_vol_pca.pkl, and full_clustering/ (Leiden states).")
print("Re-run 7.1 to copy these to Drive as well.")

---
### Notes

**Latent space versus volume space.** `analyze` clusters `z`; `analyze_landscape` clusters
voxels. The latent's geometry is an artefact of how the encoder chose to arrange things — it has
no physical units and no guarantee that equal distances mean equal structural change. Volume
space does. That is the whole reason this analysis exists, and it is why the states here can
disagree with `analyze`'s k-means clusters. When they disagree, the landscape is the one making
a structural claim.

**Occupancies are estimates, not measurements.** A particle is assigned to the state containing
the k-means volume it is nearest to in latent space. With `-N 1000` over a continuum, the state
boundaries land wherever agglomerative clustering happens to cut, and `-M` is your choice, not a
measurement. Re-run with a different `-M` and `--linkage` before quoting a number; `ward` tends
to produce more equal-sized states than `average`. If the counts move a lot, the distribution is
continuous and "states" is the wrong model for it.

**The mask matters more than anything else here.** Every distance is computed over masked
voxels, so the mask decides what the clustering is even looking at. Check `mask_slices.png`
first. If a flexible domain sits outside the mask, no amount of clustering will find it — and
`--dilate` / `--cosine-edge` are in Ångströms, so they only mean what you think if `--Apix` is
right. That is why cell 2.2 refuses to guess it.

**Cost.** Volume generation dominates, and it scales with the **cube** of `-d`: at fixed `-N`,
box 256 is 8× the disk and time of box 128. For a first look, `-N 500 -d 128` gets you most of
the picture in a fraction of the time.

**Re-clustering is cheap.** Changing only `n_states` or `linkage` does not need new volumes —
tick `skip_volume_generation` in 4.1 and it reuses `kmeans<N>/`, going straight to the PCA and
clustering. Keep `sketch_size` at whatever produced them.

**Nothing here touches the training run.** All work happens in `/content/landscape_work/`, and
only cells 6.1 and 7.1 write to Drive, both into folders of their own.